# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [5]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess


In [6]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key not set (and this is optional)
Grok API Key not set (and this is optional)
Groq API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)


In [7]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [8]:
models = ["qwen3-coder:480b-cloud", "gpt-oss:20b-cloud", "minimax-m2:cloud"]

clients = {"qwen3-coder:480b-cloud": ollama, "gpt-oss:20b-cloud": ollama, "minimax-m2:cloud": ollama}

languages = languages = ["c plus plus", "java", "javascript"]

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [9]:
!ollama pull qwen3-coder:480b-cloud

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 476b4620b85b: 100% ▕██████████████████▏  382 B                         
verifying sha256 digest 
writing manifest 
success 


In [10]:
!ollama pull gpt-oss:20b-cloud

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling cf2ed067e945: 100% ▕██████████████████▏  381 B                         
verifying sha256 digest 
writing manifest 
success 


In [11]:
!ollama pull minimax-m2:cloud

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 32677c818575: 100% ▕██████████████████▏  382 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling 32677c818575: 100% ▕██████████████████▏  382 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (x86_64-win32-seh-rev0, Built by MinGW-Builds project) 15.2.0',
   'g++': 'g++.EXE (x86_64-win32-seh-rev0, Built by MinGW-Builds project) 15.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [22]:
compile_command = ["g++.EXE", "-O3", "-std=c++17", "main.cpp", "-o", "main.exe"]
run_command = ["main.exe"]


## And now, on with the main task

In [13]:
system_prompt = """
Your task is to convert Python code into high performance code in the language user ask for.
Respond only with the code in the language user asks for. Do not provide any explanation other than occasional comments.
The converted language code response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python, language):
    # Map language names to file extensions and response formats
    language_configs = {
        "c plus plus": {
            "extension": "cpp",
            "response": "C++",
            "file": "main.cpp",
            "compile": compile_command
        },
        "java": {
            "extension": "java", 
            "response": "Java",
            "file": "Main.java",
            "compile": ["javac", "Main.java"]  # You'll need to set this up
        },
        "javascript": {
            "extension": "js",
            "response": "JavaScript", 
            "file": "main.js",
            "compile": None  # No compilation needed
        }
    }
    
    config = language_configs.get(language, language_configs["c plus plus"])
    
    return f"""
Port this Python code to {config['response']} code with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called {config['file']}
{('The compilation command is: ' + str(config['compile'])) if config['compile'] else 'No compilation needed for JavaScript.'}
Respond only with {config['response']} code.
Python code to port:

```python
{python}
```
"""

In [14]:
def messages_for(python, language):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python, language)}
    ]
 

In [15]:
def write_output(cpp_code, language): 
    language_configs = { "c plus plus": "main.cpp", "java": "Main.java", "javascript": "main.js" }
    filename = language_configs.get(language, "main.cpp")
    with open(filename, "w") as f:
        f.write(cpp_code)
    return filename


In [16]:
def port(model, python, language):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python, language), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    
    # Remove code block markers (adapted for different languages)
    reply = reply.replace('```cpp','').replace('```java','').replace('```js','').replace('```javascript','').replace('```script','').replace('```','')

    # Write to appropriate file based on language
    filename = write_output(reply, language)
    print(f"Code written to {filename}")
    return reply


In [17]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [18]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [19]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [40]:
with gr.Blocks() as ui:
    # Row 3: [Model Selection] [Convert Button] 
    with gr.Row(variant='panel'):
        # blank = gr.Textbox(visible=False)  # Invisible spacer
        header = gr.Label("Code Converter")
    
    # Row 1: [Blank] [Language Selection]
    with gr.Row():
        model = gr.Dropdown(models, label="Select LLM", value=models[0])
        language = gr.Dropdown(languages, label="Select language", value=languages[0])
    
    # Row 2: [Python Code] [Generated Code]
    with gr.Row():
        python = gr.TextArea(label="Python code:", lines=28, value=pi)
        cpp = gr.TextArea(label="Generated code:", lines=28)
    
    # Row 3: [Model Selection] [Convert Button] 
    with gr.Row():
        # blank = gr.Textbox(visible=False)  # Invisible spacer
        convert = gr.Button("Convert Code", variant="primary", )

    convert.click(port, inputs=[model, python, language], outputs=[cpp], show_progress="full")

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


In [32]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.629831 seconds

Result: 3.141592656089
Execution Time: 0.874401 seconds

Result: 3.141592656089
Execution Time: 0.546585 seconds



minimax-m2:cloud: 0.422168  
qwen3-coder:480b-cloud: 0.629831  
gpt-oss:20b-cloud: 0.416775  





In Ed's experiments, the performance speedups were:

9th place: Qwen 2.5 Coder: Fail  
8th place: OpenAI GPT-OSS 120B: 14X speedup    
7th place: DeepSeek Coder v2: 168X speedup  
6th place: Qwen3 Coder 30B: 168X speedup   
5th place: Claude Sonnet 4.5: 184X speedup   
4th place: GPT-5: 233X speedup  
**3rd place: oss-20B: 238X speedup**  
2nd place: Grok 4: 1060X speedup  
1st place: Gemini 2.5 Pro: 1440X speedup  